[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/WesleyInfoBr2/cienciadedados/blob/main/capitulos/cap03_coleta_preparacao/exemplo04_limpeza_logistica.ipynb)

# Cap 3.3: Diagnóstico de qualidade e tratamento dos registros logísticos

**Livro:** Ciência de Dados e Tomada de Decisão, Capítulo 3, Seção 3.3 Limpeza e tratamento de dados  
**Ferramenta:** Python (pandas)  
**Base:** Base didática Caso Tech, abas `logística` e `Pedidos`  
**Repositório:** [github.com/WesleyInfoBr2/cienciadedados](https://github.com/WesleyInfoBr2/cienciadedados)

## Contexto e objetivo

A aba `logística` da base Caso Tech registra, para cada pedido vendido, a data de entrega prevista, a data de entrega real, o status da entrega, a transportadora e o campo derivado `Atraso em dias`. Antes de usar esses dados para descrever prazos ou construir modelos, é preciso verificar se eles representam corretamente o processo de entrega.

O objetivo deste notebook é diagnosticar a qualidade desses registros e tratá-los de forma documentada. O diagnóstico verifica tipos, chaves, ausências, ordem das datas e consistência do campo de atraso. Em especial, investiga as datas de entrega ausentes e os atrasos negativos, distinguindo três situações que exigem decisões diferentes: **antecipação** (entrega antes do prazo), **pedido sem desfecho** (entrega ainda não realizada na data de referência da base) e **inconsistência** (valor que não pode ser explicado pelos campos de origem).

A aba `Pedidos` é usada apenas para conferir chaves e datas. A integração de tabelas para análise é tratada na Seção 3.5.

## Pré-requisitos

- Noções de pandas: leitura de arquivos, seleção de colunas, filtros e `value_counts()`.
- Bibliotecas `pandas` e `openpyxl` (já disponíveis no Google Colab). Em ambiente local: `pip install pandas openpyxl`.
- Conexão com a internet, pois a base é lida diretamente do repositório.

## Passo 1: Carregamento e inspeção da estrutura

In [ ]:
import pandas as pd

URL_BASE = (
    "https://github.com/WesleyInfoBr2/cienciadedados/raw/refs/heads/main/"
    "dados/base%20did%C3%A1tica%20Caso%20Tech.xlsx"
)

logistica = pd.read_excel(URL_BASE, sheet_name="logística")
pedidos = pd.read_excel(URL_BASE, sheet_name="Pedidos")

print(f"logística: {logistica.shape[0]} linhas x {logistica.shape[1]} colunas")
print(f"Pedidos:   {pedidos.shape[0]} linhas x {pedidos.shape[1]} colunas")
logistica.head()

logística: 15456 linhas x 6 colunas
Pedidos:   15630 linhas x 4 colunas


,Cod.Pedido,Data de Entrega Prevista,Data de Entrega Real,Status da Entrega,Transportadora,Atraso em dias
0,1,2024-09-08,2024-09-12,Entregue,LATAM Cargo,4
1,2,2024-03-05,2024-03-08,Entregue,JadLog,3
2,3,2024-02-13,2024-02-12,Entregue,Correios,-1
3,4,2023-12-20,2023-12-24,Entregue,LATAM Cargo,4
4,5,2023-12-20,2023-12-19,Entregue,Loggi,-1


In [ ]:
logistica.dtypes

Cod.Pedido                           int64
Data de Entrega Prevista    datetime64[ns]
Data de Entrega Real        datetime64[ns]
Status da Entrega                   object
Transportadora                      object
Atraso em dias                       int64
dtype: object

In [ ]:
# Categorias registradas nas variáveis qualitativas
for coluna in ["Status da Entrega", "Transportadora"]:
    print(logistica[coluna].value_counts(), "\n")

Status da Entrega
Entregue    15373
Atrasado       83
Name: count, dtype: int64 

Transportadora
LATAM Cargo    3984
Correios       3846
Loggi          3836
JadLog         3790
Name: count, dtype: int64 



**O que verificar.** As duas colunas de data devem estar no tipo `datetime64`, e `Cod.Pedido` e `Atraso em dias` em tipo inteiro. Se uma data aparecer como `object`, a leitura não reconheceu o formato e a conversão com `pd.to_datetime()` deve ser feita antes de prosseguir. As variáveis qualitativas devem conter apenas categorias esperadas, sem variações de grafia (por exemplo, "Correios" e "correios").

## Passo 2: Chaves e correspondência com a tabela de pedidos

In [ ]:
duplicados = logistica["Cod.Pedido"].duplicated().sum()
print(f"Códigos de pedido duplicados na logística: {duplicados}")

sem_pedido = ~logistica["Cod.Pedido"].isin(pedidos["Cod.Pedido"])
print(f"Registros logísticos sem pedido correspondente: {sem_pedido.sum()}")

pedidos_sem_logistica = pedidos[~pedidos["Cod.Pedido"].isin(logistica["Cod.Pedido"])]
print(f"Pedidos sem registro logístico: {len(pedidos_sem_logistica)}")
print(pedidos_sem_logistica["Status"].value_counts())

Códigos de pedido duplicados na logística: 0
Registros logísticos sem pedido correspondente: 0
Pedidos sem registro logístico: 174
Status
Cancelado    174
Name: count, dtype: int64


**Interpretação.** A chave `Cod.Pedido` é única na logística, ou seja, a unidade de observação é o pedido. Todo registro logístico tem pedido correspondente. Os pedidos sem registro logístico são exatamente os cancelados, o que é coerente com o processo: pedido cancelado não gera entrega. Essa diferença não é, portanto, uma falha de registro.

## Passo 3: Auditoria de ausências

In [ ]:
ausencias = pd.DataFrame({
    "ausentes": logistica.isna().sum(),
    "percentual": (logistica.isna().mean() * 100).round(2),
})
ausencias

,ausentes,percentual
Cod.Pedido,0,0.00
Data de Entrega Prevista,0,0.00
Data de Entrega Real,83,0.54
Status da Entrega,0,0.00
Transportadora,0,0.00
Atraso em dias,0,0.00


In [ ]:
sem_data_real = logistica[logistica["Data de Entrega Real"].isna()]
print(sem_data_real["Status da Entrega"].value_counts(), "\n")
print(sem_data_real["Atraso em dias"].describe())

Status da Entrega
Atrasado    83
Name: count, dtype: int64 

count      83.000000
mean    -3992.048193
std         1.956251
min     -3997.000000
25%     -3993.000000
50%     -3992.000000
75%     -3991.000000
max     -3989.000000
Name: Atraso em dias, dtype: float64


**Interpretação.** Somente `Data de Entrega Real` tem ausências: 83 registros (0,54%). Todos têm status "Atrasado", e nenhum registro com status "Entregue" está sem data. A ausência, portanto, não é aleatória: está associada a uma situação específica do processo.

Há, porém, um problema. O campo `Atraso em dias` está preenchido nesses 83 registros, com valores entre −3.997 e −3.989, o que equivaleria a entregas quase onze anos antes do prazo. Como não existe data de entrega real, não há atraso a calcular. Esses valores não podem ser reproduzidos a partir dos campos de origem e são tratados como **inconsistência do campo derivado**, não como antecipação.

## Passo 4: Data de referência e situação dos pedidos sem data de entrega

In [ ]:
data_referencia = logistica["Data de Entrega Real"].max()
print(f"Data de referência adotada (última entrega registrada): {data_referencia:%d/%m/%Y}")

pendentes = sem_data_real.merge(pedidos[["Cod.Pedido", "Data pedido"]], on="Cod.Pedido")
print(f"Data do pedido:   {pendentes['Data pedido'].min():%d/%m/%Y} a {pendentes['Data pedido'].max():%d/%m/%Y}")
print(f"Prazo previsto:   {pendentes['Data de Entrega Prevista'].min():%d/%m/%Y} a "
      f"{pendentes['Data de Entrega Prevista'].max():%d/%m/%Y}")
print(f"Prazos vencidos até a data de referência: "
      f"{(pendentes['Data de Entrega Prevista'] < data_referencia).sum()} de {len(pendentes)}")

Data de referência adotada (última entrega registrada): 10/10/2024
Data do pedido:   23/09/2024 a 30/09/2024
Prazo previsto:   01/10/2024 a 09/10/2024
Prazos vencidos até a data de referência: 83 de 83


**Interpretação.** A base não informa explicitamente a data em que foi extraída. Assumindo como referência a última data de entrega registrada, os 83 pedidos foram feitos no fim de setembro de 2024, tinham prazo no início de outubro e estavam com o prazo vencido nessa data. O status "Atrasado" significa, portanto, **entrega pendente após o prazo**, e não entrega realizada com atraso. São pedidos **sem desfecho**: o atraso final é desconhecido e só seria observado depois. Eles devem permanecer na base, mas não podem entrar no cálculo de prazos realizados.

## Passo 5: Ordem das datas e consistência do campo de atraso

In [ ]:
datas = logistica.merge(pedidos[["Cod.Pedido", "Data pedido"]], on="Cod.Pedido")
print("Prazo previsto anterior à data do pedido:",
      (datas["Data de Entrega Prevista"] < datas["Data pedido"]).sum())
print("Entrega real anterior à data do pedido:",
      (datas["Data de Entrega Real"] < datas["Data pedido"]).sum())

entregues = logistica.dropna(subset=["Data de Entrega Real"]).copy()
entregues["atraso_recalculado"] = (
    entregues["Data de Entrega Real"] - entregues["Data de Entrega Prevista"]
).dt.days
entregues["diferenca"] = entregues["atraso_recalculado"] - entregues["Atraso em dias"]

print(f"\nEntregas com as duas datas: {len(entregues)}")
print("Diferença entre o atraso recalculado e o campo original:")
print(entregues["diferenca"].value_counts().sort_index())

Prazo previsto anterior à data do pedido: 0
Entrega real anterior à data do pedido: 0

Entregas com as duas datas: 15373
Diferença entre o atraso recalculado e o campo original:
diferenca
-1        7
 0    15314
 1       52
Name: count, dtype: int64


In [ ]:
divergentes = entregues[entregues["diferenca"] != 0]

# Verifica se o intervalo entre as duas datas contém o dia 29/02/2024
dia_bissexto = pd.Timestamp("2024-02-29")
inicio = divergentes[["Data de Entrega Prevista", "Data de Entrega Real"]].min(axis=1)
fim = divergentes[["Data de Entrega Prevista", "Data de Entrega Real"]].max(axis=1)
contem_29fev = (inicio <= dia_bissexto) & (fim >= dia_bissexto)
print(f"Divergências cujo intervalo contém 29/02/2024: {contem_29fev.sum()} de {len(divergentes)}")

divergentes[["Cod.Pedido", "Data de Entrega Prevista", "Data de Entrega Real",
             "Atraso em dias", "atraso_recalculado"]].head()

Divergências cujo intervalo contém 29/02/2024: 59 de 59


,Cod.Pedido,Data de Entrega Prevista,Data de Entrega Real,Atraso em dias,atraso_recalculado
626,627,2024-02-26,2024-03-02,4,5
627,628,2024-02-28,2024-03-01,1,2
685,686,2024-03-01,2024-02-28,-1,-2
699,700,2024-02-28,2024-03-03,3,4
1129,1130,2024-02-28,2024-03-02,2,3


**Interpretação.** As datas estão em ordem coerente: nenhum prazo ou entrega é anterior à data do pedido. Entre as entregas realizadas, o campo `Atraso em dias` coincide com a diferença entre as datas em quase todos os casos. As 59 divergências são todas de um dia e, sem exceção, correspondem a intervalos que passam por 29/02/2024. A divergência é sistemática: o campo e as datas foram produzidos com regras de calendário diferentes quanto ao dia adicional do ano bissexto. Os dados, isoladamente, não permitem saber qual dos dois está errado. Adota-se aqui a premissa de que as datas são o registro primário do evento e de que `Atraso em dias` é um campo derivado; por isso, o atraso é recalculado a partir das datas. Essa premissa deve ser documentada e, em um contexto real, confirmada com a área responsável pelo sistema de origem.

In [ ]:
print(entregues["atraso_recalculado"].value_counts().sort_index())

atraso_recalculado
-2       7
-1    2688
 0    2563
 1    2523
 2    2553
 3    2502
 4    2517
 5      20
Name: count, dtype: int64


**Interpretação.** Entre as entregas realizadas, o atraso recalculado varia de −2 a 4 dias, com 20 casos de 5 dias. Os valores −2 e 5 surgem apenas nos registros recalculados por causa do dia 29/02. Os valores negativos são **antecipações**: entregas feitas um ou dois dias antes do prazo. Eles são legítimos e não devem ser removidos nem convertidos em zero. Um atraso negativo só é inconsistente quando não pode ser explicado pelas datas, como ocorre nos 83 pedidos sem desfecho.

## Passo 6: Tratamento e base validada

In [ ]:
logistica_tratada = logistica.copy()

# Preserva o campo original para rastreabilidade
logistica_tratada = logistica_tratada.rename(columns={"Atraso em dias": "atraso_original"})

# Atraso recalculado pelas datas; fica ausente (NaN) quando não há entrega
logistica_tratada["atraso_dias"] = (
    logistica_tratada["Data de Entrega Real"] - logistica_tratada["Data de Entrega Prevista"]
).dt.days

# Situação da entrega na data de referência
sem_desfecho = logistica_tratada["Data de Entrega Real"].isna()
atraso = logistica_tratada["atraso_dias"]
logistica_tratada["situacao_entrega"] = "Com atraso"
logistica_tratada.loc[atraso == 0, "situacao_entrega"] = "No prazo"
logistica_tratada.loc[atraso < 0, "situacao_entrega"] = "Antecipada"
logistica_tratada.loc[sem_desfecho, "situacao_entrega"] = "Sem desfecho"

# Indicador da regra aplicada a cada registro
logistica_tratada["regra_tratamento"] = "Sem alteração"
alterado = (logistica_tratada["atraso_dias"] != logistica_tratada["atraso_original"]) & ~sem_desfecho
logistica_tratada.loc[alterado, "regra_tratamento"] = "Atraso recalculado (29/02)"
logistica_tratada.loc[sem_desfecho, "regra_tratamento"] = "Atraso anulado (sem entrega)"

logistica_tratada["atraso_dias"] = logistica_tratada["atraso_dias"].astype("Int64")
logistica_tratada.head()

,Cod.Pedido,Data de Entrega Prevista,Data de Entrega Real,Status da Entrega,Transportadora,atraso_original,atraso_dias,situacao_entrega,regra_tratamento
0,1,2024-09-08,2024-09-12,Entregue,LATAM Cargo,4,4,Com atraso,Sem alteração
1,2,2024-03-05,2024-03-08,Entregue,JadLog,3,3,Com atraso,Sem alteração
2,3,2024-02-13,2024-02-12,Entregue,Correios,-1,-1,Antecipada,Sem alteração
3,4,2023-12-20,2023-12-24,Entregue,LATAM Cargo,4,4,Com atraso,Sem alteração
4,5,2023-12-20,2023-12-19,Entregue,Loggi,-1,-1,Antecipada,Sem alteração


In [ ]:
resumo = pd.crosstab(logistica_tratada["situacao_entrega"],
                     logistica_tratada["regra_tratamento"], margins=True, margins_name="Total")
resumo

regra_tratamento,Atraso anulado (sem entrega),Atraso recalculado (29/02),Sem alteração,Total
situacao_entrega,,,,
Antecipada,0,7,2688,2695
Com atraso,0,52,10063,10115
No prazo,0,0,2563,2563
Sem desfecho,83,0,0,83
Total,83,59,15314,15456


In [ ]:
# Verificações finais
assert logistica_tratada["Cod.Pedido"].is_unique
assert logistica_tratada.loc[sem_desfecho, "atraso_dias"].isna().all()
assert (logistica_tratada.loc[~sem_desfecho, "atraso_dias"].between(-2, 5)).all()

logistica_tratada.to_csv("logistica_tratada.csv", index=False, encoding="utf-8-sig")
print("Base validada gravada em logistica_tratada.csv")

Base validada gravada em logistica_tratada.csv


## Interpretação dos resultados

Dos 15.456 registros logísticos, 15.314 foram mantidos sem alteração. Os outros 142 receberam tratamento por dois motivos distintos. Nos 59 registros cujo intervalo passa por 29/02/2024, o atraso foi recalculado a partir das datas, sob a premissa de que elas são o registro primário. Nos 83 pedidos sem desfecho, o atraso foi anulado, porque não existe entrega a medir. Nenhum registro foi excluído.

A variável `situacao_entrega` separa explicitamente as entregas antecipadas, no prazo e com atraso dos pedidos sem desfecho. Essa separação é necessária nas etapas seguintes: estatísticas de prazo realizado devem usar apenas entregas concluídas, e qualquer variável-alvo derivada do atraso deve tratar os pedidos sem desfecho como observações sem resposta conhecida. A data de referência adotada deve ser registrada como premissa do tratamento.

## Erros comuns

- **Tratar todo atraso negativo como erro.** Antecipações de um ou dois dias são legítimas; o que caracteriza a inconsistência é a impossibilidade de reproduzir o valor a partir das datas.
- **Excluir os registros sem data de entrega.** Eles representam pedidos pendentes. Excluí-los subestima o volume de entregas em atraso.
- **Preencher a data ausente com a data prevista, a data de referência ou zero.** Qualquer imputação cria uma entrega que não ocorreu.
- **Confiar no campo derivado sem recalculá-lo.** Campos calculados podem conter erros sistemáticos, como a divergência associada ao ano bissexto, que só aparecem quando comparados com os campos de origem.
- **Sobrescrever o campo original.** Manter `atraso_original` e `regra_tratamento` permite auditar cada decisão.